In [ ]:
'''
code qui parcoure les fichiers .json et supprime le répertoire si il n'y a pas de fichiers de
modélisation dans model/ et qui génère des stats par modèles (.csv) avec le nombre de fichiers
par extensions (ex : .csv, .omex, etc.) 
'''

import os
import shutil
import csv
from pathlib import Path

# Configuration du répertoire racine (ajustez le nom si nécessaire)
racine_data = Path("./BioModels_Data_cleaned_redirection_metadata_json_dico_")

# Liste des extensions considérées comme "fichiers de modélisation"
# On exclut les images, les notes de curation et les manifests
EXTENSIONS_MODELE = {
    '.xml', '.sbml', '.omex', '.sedml', '.cps', '.m', '.ode', 
    '.py', '.f', '.java', '.vcml', '.zip', '.tsv', '.yaml', '.graphml'
}

stats_globales = []

print("Début du nettoyage et de l'analyse...")

# On parcourt les dossiers de maladies (ex: covid_files)
for dossier_maladie in racine_data.iterdir():
    if not dossier_maladie.is_dir():
        continue
        
    # On parcourt chaque dossier de modèle (ex: BIOMD...)
    for dossier_model in dossier_maladie.iterdir():
        if not dossier_model.is_dir():
            continue
            
        chemin_model = dossier_model / "model"
        model_id = dossier_model.name
        
        # 1. Analyse des extensions dans le dossier 'model'
        comptage_extensions = {}
        a_un_modele = False
        
        if chemin_model.exists():
            for fichier in chemin_model.iterdir():
                if fichier.is_file():
                    ext = fichier.suffix.lower() if fichier.suffix else "no_ext"
                    comptage_extensions[ext] = comptage_extensions.get(ext, 0) + 1
                    
                    if ext in EXTENSIONS_MODELE:
                        a_un_modele = True
        
        # 2. Condition de suppression
        # Si le dossier 'model' est vide ou ne contient que des fichiers de curation/manifest
        if not a_un_modele:
            print(f"[-] Suppression de {model_id} : Aucun fichier de modélisation trouvé.")
            shutil.rmtree(dossier_model)
            continue

        # 3. Collecte des stats si le modèle est conservé
        stats_row = {
            "category": dossier_maladie.name,
            "model_id": model_id
        }
        stats_row.update(comptage_extensions)
        stats_globales.append(stats_row)

# 4. Génération du fichier CSV
if stats_globales:
    # Récupération de toutes les extensions uniques pour les colonnes du CSV
    toutes_colonnes = set()
    for row in stats_globales:
        toutes_colonnes.update(row.keys())
    
    # Tri des colonnes pour avoir category et model_id au début
    colonnes_ordonnees = ['category', 'model_id'] + sorted([c for c in toutes_colonnes if c not in ['category', 'model_id']])
    
    chemin_csv = racine_data / "statistiques_modeles_nettoyes.csv"
    with open(chemin_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=colonnes_ordonnees)
        writer.writeheader()
        # On remplace les valeurs absentes (None) par 0
        for row in stats_globales:
            row_final = {col: row.get(col, 0) for col in colonnes_ordonnees}
            writer.writerow(row_final)
            
    print(f"\nTerminé !")
    print(f"Statistiques générées dans : {chemin_csv}")
else:
    print("\nAucun modèle valide trouvé.")